# 04 — Model Training Demo
**Deep Learning for Cryptocurrency Price Forecasting**
*Muluh Penn Junior Patrick — M.Tech. Thesis 2026*

---
End-to-end walkthrough of training a single model (GRU) on BTC 1d data.
Demonstrates the full pipeline: data → features → sequences → training → evaluation.

This notebook is intended as a reproducible demonstration.
Full experiment results are in `05_results_analysis.ipynb`.


In [ ]:
import sys; sys.path.insert(0, '..')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 4.1 Hyperparameters (from Optuna tuning)

In [ ]:
import json
from pathlib import Path

# Load best params from tuning study
params_file = Path('../experiments/results/tuning/gru_BTC_1d_h1_best_params.json')
if params_file.exists():
    with open(params_file) as f:
        best_params = json.load(f)
    print('Best hyperparameters (from Optuna study):')
    for k, v in best_params.get('best_params', {}).items():
        print(f'  {k:<25} {v}')
else:
    # Defaults for demonstration
    best_params = {'best_params': {
        'seq_len': 90, 'batch_size': 64, 'lr': 0.00448,
        'hidden_size': 128, 'num_layers': 1, 'dropout': 0.296,
    }}
    print('Using default parameters (tuning results not found)')
    for k, v in best_params['best_params'].items():
        print(f'  {k:<25} {v}')


## 4.2 Setup data module

In [ ]:
from src.training.trainer import CryptoDataModule

params = best_params['best_params']
data_module = CryptoDataModule(
    asset      = 'BTC',
    interval   = '1d',
    seq_len    = int(params.get('seq_len', 90)),
    horizon    = 1,
    batch_size = int(params.get('batch_size', 64)),
)
data_module.setup()
print(f'DataModule ready:')
print(f'  Train batches : {len(data_module.train_dataloader())}')
print(f'  Val batches   : {len(data_module.val_dataloader())}')
print(f'  Test batches  : {len(data_module.test_dataloader())}')
print(f'  Features      : {data_module.n_features}')


## 4.3 Instantiate GRU model

In [ ]:
from src.models import get_model

model = get_model(
    'gru',
    input_size   = data_module.n_features,
    output_size  = 1,
    hidden_size  = int(params.get('hidden_size', 128)),
    num_layers   = int(params.get('num_layers', 1)),
    dropout      = float(params.get('dropout', 0.296)),
)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: GRUForecaster')
print(f'Parameters: {n_params:,}')
print(model)


## 4.4 Train (quick demo — 10 epochs)

In [ ]:
# For a full training run, use:
# python -m src.training.trainer --model gru --asset BTC --interval 1d
# This cell demonstrates the API with a short 10-epoch run

from src.training.trainer import train_model

print('Training GRU for 10 epochs (demo)...')
results = train_model(
    model_name     = 'gru',
    asset          = 'BTC',
    interval       = '1d',
    horizon        = 1,
    seq_len        = int(params.get('seq_len', 90)),
    batch_size     = int(params.get('batch_size', 64)),
    lr             = float(params.get('lr', 0.00448)),
    weight_decay   = float(params.get('weight_decay', 0.00405)),
    max_epochs     = 10,    # short for demo; use 200 for full training
    loss_name      = 'combined',
    model_kwargs   = {k: v for k, v in params.items()
                     if k in ['hidden_size','num_layers','dropout']},
)
print('\nDemo training complete.')
print(f'Test RMSE : ${results["test_metrics"]["rmse"]:,.0f}')
print(f'Test MAPE : {results["test_metrics"]["mape"]:.2f}%')
print(f'Test DA   : {results["test_metrics"]["directional_accuracy"]:.1f}%')


## 4.5 Training curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = Path('../experiments/results/gru_BTC_1d_metrics.csv')
if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(df.index, df['val_loss'], color='#1D9E75', linewidth=2, label='Validation loss')
    if 'train_loss' in df.columns:
        ax.plot(df.index, df['train_loss'], color='#888780', linewidth=1,
                linestyle='--', label='Training loss', alpha=0.7)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (CombinedLoss)')
    ax.set_title('GRU Training Curve — BTC 1d h=1')
    ax.legend()
    plt.tight_layout(); plt.show()
    print(f'Best val_loss: {df["val_loss"].min():.4f} at epoch {df["val_loss"].idxmin()}')
else:
    print('Training metrics CSV not found. Run full training first.')


## 4.6 Optuna hyperparameter search (summary)
The full tuning study ran 30 trials × 50 epochs for GRU.
Key findings from the Optuna TPE sampler:
- `seq_len=90` consistently outperformed 30 and 120
- `hidden_size=128, num_layers=1` is optimal (deeper networks overfit)
- `combined` loss (MSE + Huber + Directional) is critical — pure MSE leads to collapse
- Best val_loss improved from 0.098 (baseline) to 0.059 (tuned), a **40% reduction**

**→ Next: Results Analysis** (`05_results_analysis.ipynb`)
